# TP3 — ETo, VPD e índices de estrés hídrico en Casablanca

## Validación en Colliguay y aplicación del enfoque de Zambrano et al. (2025)

**Pregunta guía:** ¿cómo representan ERA5-Land y una estación meteorológica la demanda atmosférica, y cómo se relacionan la ETo y diferentes estimaciones del VPD con los índices de humedad y estrés derivados de Sentinel-2?

El trabajo tiene dos territorios y dos funciones diferentes:

1. **Colliguay:** referencia observacional para evaluar ETo y VPD derivados de ERA5-Land.
2. **Casablanca:** área de aplicación, donde se integran ERA5-Land y Sentinel-2.

El cuaderno no aplica Machine Learning. Su producto final es una tabla temporal documentada que podrá utilizarse en la TP4.

## Resultados de aprendizaje

Al finalizar podrá:

1. reconocer las variables, unidades, resolución y limitaciones de ERA5-Land;
2. calcular ETo diaria mediante FAO-56 Penman–Monteith;
3. calcular VPD con temperatura, humedad relativa y punto de rocío;
4. comparar tres definiciones de VPD en una estación y en ERA5-Land;
5. validar estimaciones de ERA5-Land con observaciones de Colliguay;
6. aplicar el procesamiento a la estación y área monitoreada de Casablanca;
7. calcular MSI, DWSI, NDMI y NDVI con Sentinel-2;
8. analizar relaciones contemporáneas y antecedentes de 3 y 7 días;
9. interpretar correlaciones considerando escala espacial, estacionalidad y ausencia de causalidad.

## 1. ¿Qué es ERA5-Land?

ERA5-Land es un **reanálisis**: combina un modelo físico de la superficie terrestre con observaciones asimiladas para generar series espacial y temporalmente continuas. No equivale a una estación meteorológica.

| Fuente | Representación | Escala aproximada en esta actividad |
|---|---|---|
| Estación Colliguay | Medición local en un punto | entorno inmediato del sensor |
| ERA5-Land | Promedio/modelación de una celda | píxel de aproximadamente 11 km |
| Sentinel-2 | Reflectancia de la superficie | 10–20 m según la banda |

La comparación punto–píxel contiene una diferencia de soporte espacial. Una discrepancia puede deberse tanto al modelo como a la topografía, exposición, distancia, manejo local o representatividad de la estación.

**Colección:** `ECMWF/ERA5_LAND/DAILY_AGGR`  
**Documentación:** https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_DAILY_AGGR

## 2. Configuración manual

Edite solamente las líneas indicadas. Las fechas finales son **exclusivas** en Earth Engine. El período largo se usa para validar en Colliguay; el período corto se usa para analizar Casablanca y Sentinel-2.

In [ ]:
# EDITE ESTAS LÍNEAS
PROJECT_ID = 'lucasvituri'
CARPETA_DRIVE = '/content/drive/MyDrive/02_TP3_ZAMBRANO_SWIR_ETO_VPD_CASABLANCA'

# Validación de ERA5-Land frente a Colliguay
VALIDACION_INICIO = '2017-08-06'
VALIDACION_FIN_EXCLUSIVA = '2026-08-21'

# Aplicación en Casablanca y cruce con Sentinel-2
ANALISIS_INICIO = '2023-09-28'
ANALISIS_FIN_EXCLUSIVA = '2024-03-31'
NUBOSIDAD_MAX = 30

# Fecha usada solamente en el mapa meteorológico
FECHA_MAPA = '2024-01-15'

# Control de calidad mínimo de los datos diarios de la estación
COMPLETITUD_MINIMA = 80

## 3. Preparar Colab, Drive y Earth Engine

El mapa se crea sin OpenStreetMap para evitar bloqueos 403 del servidor de teselas. Las capas científicas de Earth Engine continúan funcionando normalmente.

In [ ]:
%pip install -q geemap geopandas pyogrio seaborn

In [ ]:
from pathlib import Path
import warnings

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from google.colab import drive

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
sns.set_theme(style='whitegrid', context='notebook')

drive.mount('/content/drive')

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

CARPETA_DRIVE = Path(CARPETA_DRIVE)
RUTA_COLLIGUAY = CARPETA_DRIVE / '03_DATOS' / 'COLLIGUAY_AGROMET_DIARIO.csv'
RUTA_AREA = CARPETA_DRIVE / '03_DATOS' / 'AREA_MONITOREADA_CASABLANCA.gpkg'
RUTA_SITIO = CARPETA_DRIVE / '03_DATOS' / 'SITIO_CASABLANCA.csv'
CARPETA_SALIDA = CARPETA_DRIVE / '04_RESULTADOS'
CARPETA_SALIDA.mkdir(exist_ok=True)

for ruta in [RUTA_COLLIGUAY, RUTA_AREA, RUTA_SITIO]:
    if not ruta.exists():
        raise FileNotFoundError(f'No se encontró: {ruta}')

print('Earth Engine inicializado y archivos encontrados.')

## 4. Leer las áreas y los datos de Colliguay

Colliguay está aproximadamente a 20 km del punto de Casablanca. Esta proximidad favorece la comparación regional, pero las dos localidades difieren en elevación y condiciones locales.

El archivo compacto fue preparado desde la red Agromet de Chile. Conserva el valor diario y el porcentaje de datos usados para construir cada resumen.

In [ ]:
colliguay = pd.read_csv(RUTA_COLLIGUAY, parse_dates=['fecha']).sort_values('fecha')
area = gpd.read_file(RUTA_AREA).to_crs(epsg=4326)
sitio_df = pd.read_csv(RUTA_SITIO)

lat_colliguay = float(colliguay['latitud'].iloc[0])
lon_colliguay = float(colliguay['longitud'].iloc[0])
elev_colliguay = float(colliguay['elevacion_m'].iloc[0])
lat_casablanca = float(sitio_df['latitud'].iloc[0])
lon_casablanca = float(sitio_df['longitud'].iloc[0])
elev_casablanca = float(sitio_df['elevacion_m'].iloc[0])

aoi_fc = geemap.gdf_to_ee(area)
aoi = aoi_fc.geometry()
punto_colliguay = ee.Geometry.Point([lon_colliguay, lat_colliguay])
punto_casablanca = ee.Geometry.Point([lon_casablanca, lat_casablanca])

def distancia_haversine_km(lat1, lon1, lat2, lon2):
    radio = 6371.0
    p1, p2 = np.radians([lat1, lat2])
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2)**2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2)**2
    return 2 * radio * np.arcsin(np.sqrt(a))

distancia = distancia_haversine_km(
    lat_colliguay, lon_colliguay, lat_casablanca, lon_casablanca
)
print('Registros de Colliguay:', len(colliguay))
print('Período:', colliguay['fecha'].min().date(), 'a', colliguay['fecha'].max().date())
print(f'Distancia Colliguay–Casablanca: {distancia:.1f} km')
display(colliguay.head())

In [ ]:
mapa_sitios = geemap.Map(basemap={})
mapa_sitios.centerObject(aoi.union(punto_colliguay.buffer(1000)), 10)
mapa_sitios.addLayer(
    aoi_fc.style(color='FFD700', fillColor='00000000', width=3), {},
    'Área de Casablanca'
)
mapa_sitios.addLayer(punto_casablanca, {'color': 'FF0000'}, 'Estación Casablanca')
mapa_sitios.addLayer(punto_colliguay, {'color': '0066FF'}, 'Estación Colliguay')
mapa_sitios

### Control de calidad

Para cada variable se revisan cobertura, valores ausentes y completitud. Un día puede ser adecuado para una variable e insuficiente para otra. Por eso el filtro de calidad se aplica según las variables necesarias en cada cálculo.

In [ ]:
variables_estacion = [
    'ET0_estacion_mm_dia', 'Tmed_estacion_C', 'Tmin_estacion_C',
    'Tmax_estacion_C', 'HRmed_estacion_pct', 'HRmin_estacion_pct',
    'HRmax_estacion_pct', 'Radiacion_estacion_MJ_m2_dia',
    'Presion_estacion_hPa', 'Viento_estacion_km_h',
]
for columna in variables_estacion:
    colliguay[columna] = pd.to_numeric(colliguay[columna], errors='coerce')

control = pd.DataFrame({
    'variable': variables_estacion,
    'n_validos': [colliguay[c].notna().sum() for c in variables_estacion],
    'faltantes': [colliguay[c].isna().sum() for c in variables_estacion],
    'minimo': [colliguay[c].min() for c in variables_estacion],
    'maximo': [colliguay[c].max() for c in variables_estacion],
})
display(control.round(2))

plt.figure(figsize=(12, 3.8))
plt.plot(colliguay['fecha'], colliguay['ET0_estacion_mm_dia'], color='#d95f02', lw=0.8)
plt.ylabel('ETo (mm/día)')
plt.xlabel('Fecha')
plt.title('ETo observada en Colliguay')
plt.tight_layout()
plt.show()

print('Observe la estacionalidad y verifique si existen interrupciones prolongadas.')

## 5. Presión de vapor y métodos de VPD

La presión de vapor de saturación a una temperatura \(T\), expresada en °C, es:

\[
e^o(T)=0{,}6108\exp\left(\frac{17{,}27T}{T+237{,}3}\right)
\]

El **déficit de presión de vapor** es la diferencia entre la presión de saturación \(e_s\) y la presión real de vapor \(e_a\):

\[
VPD=e_s-e_a
\]

Calcularemos tres descriptores diarios:

1. **VPD medio:** usa temperatura y humedad relativas medias.
2. **VPD FAO:** combina temperaturas máxima/mínima y humedades máxima/mínima.
3. **VPD máximo aproximado:** combina temperatura máxima y humedad mínima; representa el momento de mayor demanda atmosférica diaria.

La estación no mide VPD directamente. Todos estos valores son estimaciones derivadas de sus sensores de temperatura y humedad.

In [ ]:
def presion_saturacion(temp_c):
    return 0.6108 * np.exp(17.27 * temp_c / (temp_c + 237.3))

def agregar_vpd_estacion(df):
    resultado = df.copy()
    es_med = presion_saturacion(resultado['Tmed_estacion_C'])
    resultado['VPD_medio_estacion_kPa'] = (
        es_med * (1 - resultado['HRmed_estacion_pct'] / 100)
    ).clip(lower=0)

    es = (
        presion_saturacion(resultado['Tmax_estacion_C'])
        + presion_saturacion(resultado['Tmin_estacion_C'])
    ) / 2
    ea = (
        presion_saturacion(resultado['Tmin_estacion_C'])
        * resultado['HRmax_estacion_pct'] / 100
        + presion_saturacion(resultado['Tmax_estacion_C'])
        * resultado['HRmin_estacion_pct'] / 100
    ) / 2
    resultado['VPD_FAO_estacion_kPa'] = (es - ea).clip(lower=0)
    resultado['VPD_max_estacion_kPa'] = (
        presion_saturacion(resultado['Tmax_estacion_C'])
        * (1 - resultado['HRmin_estacion_pct'] / 100)
    ).clip(lower=0)
    return resultado

colliguay = agregar_vpd_estacion(colliguay)
columnas_vpd_estacion = [
    'VPD_medio_estacion_kPa', 'VPD_FAO_estacion_kPa', 'VPD_max_estacion_kPa'
]
display(colliguay[['fecha'] + columnas_vpd_estacion].head())

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5))
for columna, color in zip(
    columnas_vpd_estacion, ['#1b9e77', '#7570b3', '#d95f02']
):
    ejes[0].plot(colliguay['fecha'], colliguay[columna], lw=0.8,
                 label=columna.replace('_estacion_kPa', ''), color=color)
    sns.kdeplot(data=colliguay, x=columna, ax=ejes[1], label=columna.split('_estacion')[0], color=color)
ejes[0].set_title('Evolución diaria de los métodos de VPD')
ejes[0].set_ylabel('VPD (kPa)'); ejes[0].legend()
ejes[1].set_title('Distribución de los métodos'); ejes[1].set_xlabel('VPD (kPa)'); ejes[1].legend()
plt.tight_layout(); plt.show()

print('El VPD máximo debe ser mayor que los descriptores medios en la mayoría de los días.')

## 6. ETo FAO-56 Penman–Monteith con ERA5-Land

La evapotranspiración de referencia expresa la demanda atmosférica sobre una superficie de referencia:

\[
ET_o=\frac{0{,}408\Delta(R_n-G)+\gamma\frac{900}{T+273}u_2(e_s-e_a)}
{\Delta+\gamma(1+0{,}34u_2)}
\]

Donde \(R_n\) es radiación neta, \(G\) es flujo de calor del suelo —aproximado como cero en escala diaria—, \(u_2\) es viento a 2 m, \(\Delta\) es la pendiente de la curva de presión de vapor y \(\gamma\) es la constante psicrométrica.

Conversión de unidades aplicada:

- temperatura: K − 273,15 → °C;
- radiación: J m⁻² ÷ 10⁶ → MJ m⁻² día⁻¹;
- presión: Pa ÷ 1000 → kPa;
- viento: componentes a 10 m convertidas a velocidad a 2 m;
- elevación: SRTM de 30 m para estimar radiación de cielo despejado.

In [ ]:
ERA5_BANDAS = [
    'temperature_2m', 'temperature_2m_min', 'temperature_2m_max',
    'dewpoint_temperature_2m', 'dewpoint_temperature_2m_min',
    'dewpoint_temperature_2m_max', 'u_component_of_wind_10m',
    'v_component_of_wind_10m', 'surface_solar_radiation_downwards_sum',
    'surface_pressure',
]
ESCALA_ERA5_M = 11132
DEM = ee.Image('USGS/SRTMGL1_003').select('elevation').unmask(0)

def es_ee(temp_c):
    return temp_c.multiply(17.27).divide(temp_c.add(237.3)).exp().multiply(0.6108)

def agregar_eto_vpd_era5(imagen):
    imagen = ee.Image(imagen)
    tmed = imagen.select('temperature_2m').subtract(273.15)
    tmin = imagen.select('temperature_2m_min').subtract(273.15)
    tmax = imagen.select('temperature_2m_max').subtract(273.15)
    tdmed = imagen.select('dewpoint_temperature_2m').subtract(273.15)
    tdmin = imagen.select('dewpoint_temperature_2m_min').subtract(273.15)

    es_med = es_ee(tmed)
    es_fao = es_ee(tmax).add(es_ee(tmin)).divide(2)
    ea_td = es_ee(tdmed).min(es_fao)
    vpd_medio = es_med.subtract(es_ee(tdmed)).max(0).rename('VPD_medio_ERA5_kPa')
    vpd_fao = es_fao.subtract(ea_td).max(0).rename('VPD_FAO_ERA5_kPa')
    vpd_max = es_ee(tmax).subtract(es_ee(tdmin)).max(0).rename('VPD_max_ERA5_kPa')

    delta = es_med.multiply(4098).divide(tmed.add(237.3).pow(2))
    presion_kpa = imagen.select('surface_pressure').divide(1000)
    gamma = presion_kpa.multiply(0.000665)
    viento_2m = (
        imagen.select('u_component_of_wind_10m').pow(2)
        .add(imagen.select('v_component_of_wind_10m').pow(2)).sqrt()
        .multiply(0.748)
    )
    rs = imagen.select('surface_solar_radiation_downwards_sum').divide(1e6)

    latitud = ee.Image.pixelLonLat().select('latitude').multiply(np.pi / 180)
    doy = ee.Image.constant(imagen.date().getRelative('day', 'year').add(1)).toFloat()
    dr = doy.multiply(2 * np.pi / 365).cos().multiply(0.033).add(1)
    declinacion = doy.multiply(2 * np.pi / 365).subtract(1.39).sin().multiply(0.409)
    argumento = latitud.tan().multiply(declinacion.tan()).multiply(-1).clamp(-1, 1)
    angulo_solar = argumento.acos()
    ra = (
        angulo_solar.multiply(latitud.sin()).multiply(declinacion.sin())
        .add(latitud.cos().multiply(declinacion.cos()).multiply(angulo_solar.sin()))
        .multiply(dr).multiply(24 * 60 / np.pi * 0.0820)
    )
    rso = DEM.multiply(2e-5).add(0.75).multiply(ra)
    rns = rs.multiply(1 - 0.23)
    razon_radiacion = rs.divide(rso.max(1e-6)).clamp(0, 1)
    rnl = (
        tmax.add(273.16).pow(4).add(tmin.add(273.16).pow(4)).divide(2)
        .multiply(4.903e-9)
        .multiply(ea_td.max(0).sqrt().multiply(-0.14).add(0.34))
        .multiply(razon_radiacion.multiply(1.35).subtract(0.35))
    )
    rn = rns.subtract(rnl)
    numerador = (
        delta.multiply(rn).multiply(0.408)
        .add(gamma.multiply(900).divide(tmed.add(273))
             .multiply(viento_2m).multiply(es_fao.subtract(ea_td)))
    )
    denominador = delta.add(gamma.multiply(viento_2m.multiply(0.34).add(1)))
    eto = numerador.divide(denominador).max(0).rename('ETo_ERA5_mm_dia')

    diagnostico = ee.Image.cat([
        eto, vpd_medio, vpd_fao, vpd_max,
        tmed.rename('Tmed_ERA5_C'), tmin.rename('Tmin_ERA5_C'),
        tmax.rename('Tmax_ERA5_C'), tdmed.rename('Troc_medio_ERA5_C'),
        tdmin.rename('Troc_min_ERA5_C'), viento_2m.rename('Viento2m_ERA5_m_s'),
        rs.rename('Radiacion_ERA5_MJ_m2_dia'), presion_kpa.rename('Presion_ERA5_kPa'),
    ]).toFloat()
    return diagnostico.copyProperties(imagen, ['system:time_start'])

print('Funciones de ETo y VPD para ERA5-Land preparadas.')

## 7. Extraer ERA5-Land por períodos anuales

La extracción se divide por año para evitar solicitudes demasiado grandes. En Colliguay se toma el valor de la celda ERA5-Land que contiene la estación. En Casablanca se calcula la media de las celdas que intersectan el área monitoreada.

In [ ]:
SALIDAS_ERA5 = [
    'ETo_ERA5_mm_dia', 'VPD_medio_ERA5_kPa', 'VPD_FAO_ERA5_kPa',
    'VPD_max_ERA5_kPa', 'Tmed_ERA5_C', 'Tmin_ERA5_C', 'Tmax_ERA5_C',
    'Troc_medio_ERA5_C', 'Troc_min_ERA5_C', 'Viento2m_ERA5_m_s',
    'Radiacion_ERA5_MJ_m2_dia', 'Presion_ERA5_kPa',
]

def extraer_era5(geometria, fecha_inicio, fecha_fin_exclusiva, reductor):
    inicio = pd.Timestamp(fecha_inicio)
    fin = pd.Timestamp(fecha_fin_exclusiva)
    fragmentos = []
    for anio in range(inicio.year, fin.year + 1):
        tramo_inicio = max(inicio, pd.Timestamp(f'{anio}-01-01'))
        tramo_fin = min(fin, pd.Timestamp(f'{anio + 1}-01-01'))
        if tramo_inicio >= tramo_fin:
            continue
        coleccion = (
            ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
            .filterDate(tramo_inicio.strftime('%Y-%m-%d'), tramo_fin.strftime('%Y-%m-%d'))
            .select(ERA5_BANDAS).map(agregar_eto_vpd_era5)
        )

        def resumir(imagen):
            imagen = ee.Image(imagen)
            valores = imagen.select(SALIDAS_ERA5).reduceRegion(
                reducer=reductor, geometry=geometria, scale=ESCALA_ERA5_M,
                bestEffort=True, maxPixels=1e7,
            )
            return ee.Feature(None, valores).set(
                'fecha', imagen.date().format('YYYY-MM-dd')
            )

        fc = ee.FeatureCollection(coleccion.map(resumir))
        parte = geemap.ee_to_df(fc)
        if not parte.empty:
            fragmentos.append(parte)
            print(anio, '→', len(parte), 'días')

    if not fragmentos:
        raise RuntimeError('ERA5-Land no devolvió registros para el período solicitado.')
    datos = pd.concat(fragmentos, ignore_index=True)
    datos['fecha'] = pd.to_datetime(datos['fecha'])
    for columna in SALIDAS_ERA5:
        datos[columna] = pd.to_numeric(datos[columna], errors='coerce')
    return datos.sort_values('fecha').drop_duplicates('fecha')

## 8. Validar ETo y VPD de ERA5-Land en Colliguay

La ETo de estación es una variable publicada por Agromet y permite una comparación directa con la ETo calculada desde ERA5-Land. Para VPD, comparamos **estimaciones equivalentes**:

| Estación | ERA5-Land | Interpretación |
|---|---|---|
| VPD medio con T media y HR media | VPD medio con T media y punto de rocío medio | condición media diaria |
| VPD FAO con extremos de T y HR | VPD con extremos de T y punto de rocío medio | promedio diario más sensible a extremos |
| VPD máximo con Tmax y HRmin | VPD máximo con Tmax y punto de rocío mínimo | mayor demanda atmosférica aproximada |

Esta es una comparación de métodos y fuentes, no una medición instrumental directa de VPD.

In [ ]:
era5_colliguay = extraer_era5(
    punto_colliguay, VALIDACION_INICIO, VALIDACION_FIN_EXCLUSIVA, ee.Reducer.first()
)

validacion = (
    colliguay.merge(era5_colliguay, on='fecha', how='inner')
    .sort_values('fecha')
)

pares_calidad = {
    'ETo': ('ET0_estacion_mm_dia', 'ETo_ERA5_mm_dia', ['completitud_ET0_pct']),
    'VPD_medio': (
        'VPD_medio_estacion_kPa', 'VPD_medio_ERA5_kPa',
        ['completitud_TA_AVG_pct', 'completitud_HR_AVG_pct']
    ),
    'VPD_FAO': (
        'VPD_FAO_estacion_kPa', 'VPD_FAO_ERA5_kPa',
        ['completitud_TA_MIN_pct', 'completitud_TA_MAX_pct',
         'completitud_HR_MIN_pct', 'completitud_HR_MAX_pct']
    ),
    'VPD_max': (
        'VPD_max_estacion_kPa', 'VPD_max_ERA5_kPa',
        ['completitud_TA_MAX_pct', 'completitud_HR_MIN_pct']
    ),
}

print('Días coincidentes antes del control de calidad:', len(validacion))
display(validacion[['fecha', 'ET0_estacion_mm_dia', 'ETo_ERA5_mm_dia'] +
                   columnas_vpd_estacion +
                   ['VPD_medio_ERA5_kPa', 'VPD_FAO_ERA5_kPa', 'VPD_max_ERA5_kPa']].head())

In [ ]:
def metricas(observado, estimado):
    observado = np.asarray(observado, dtype=float)
    estimado = np.asarray(estimado, dtype=float)
    valido = np.isfinite(observado) & np.isfinite(estimado)
    observado, estimado = observado[valido], estimado[valido]
    if len(observado) < 3:
        return dict(n=len(observado), r=np.nan, R2=np.nan, MAE=np.nan,
                    RMSE=np.nan, sesgo=np.nan, PBIAS=np.nan, KGE=np.nan)
    r = np.corrcoef(observado, estimado)[0, 1]
    mae = np.mean(np.abs(estimado - observado))
    rmse = np.sqrt(np.mean((estimado - observado) ** 2))
    sesgo = np.mean(estimado - observado)
    pbias = 100 * np.sum(estimado - observado) / np.sum(observado)
    alpha = np.std(estimado, ddof=0) / np.std(observado, ddof=0)
    beta = np.mean(estimado) / np.mean(observado)
    kge = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
    return dict(n=len(observado), r=r, R2=r**2, MAE=mae, RMSE=rmse,
                sesgo=sesgo, PBIAS=pbias, KGE=kge)

filas_metricas = []
datos_por_par = {}
for nombre, (obs, est, columnas_calidad) in pares_calidad.items():
    mascara_calidad = validacion[columnas_calidad].ge(COMPLETITUD_MINIMA).all(axis=1)
    subconjunto = validacion.loc[
        mascara_calidad, ['fecha', obs, est]
    ].dropna()
    datos_por_par[nombre] = subconjunto
    fila = metricas(subconjunto[obs], subconjunto[est])
    fila['comparacion'] = nombre
    filas_metricas.append(fila)

tabla_metricas = pd.DataFrame(filas_metricas).set_index('comparacion').round(3)
display(tabla_metricas)

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(11, 10))
for eje, (nombre, (obs, est, _)) in zip(ejes.ravel(), pares_calidad.items()):
    datos = datos_por_par[nombre]
    sns.scatterplot(data=datos, x=obs, y=est, s=18, alpha=0.35, ax=eje)
    minimo = min(datos[obs].min(), datos[est].min())
    maximo = max(datos[obs].max(), datos[est].max())
    eje.plot([minimo, maximo], [minimo, maximo], 'k--', lw=1, label='Línea 1:1')
    m = tabla_metricas.loc[nombre]
    eje.set_title(f'{nombre}: R²={m.R2:.2f}, RMSE={m.RMSE:.2f}, PBIAS={m.PBIAS:+.1f}%')
    eje.legend()
plt.tight_layout(); plt.show()

print('Una correlación alta no garantiza ausencia de sesgo ni igualdad de magnitudes.')

In [ ]:
fig, ejes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
periodo_grafico = validacion.query("fecha >= '2023-09-01' and fecha < '2024-04-01'")
ejes[0].plot(periodo_grafico['fecha'], periodo_grafico['ET0_estacion_mm_dia'],
             color='black', label='ETo Colliguay')
ejes[0].plot(periodo_grafico['fecha'], periodo_grafico['ETo_ERA5_mm_dia'],
             color='#d95f02', label='ETo ERA5-Land')
ejes[0].set_ylabel('ETo (mm/día)'); ejes[0].legend(); ejes[0].set_title('Ejemplo de evolución diaria')

ejes[1].plot(periodo_grafico['fecha'], periodo_grafico['VPD_FAO_estacion_kPa'],
             color='black', label='VPD FAO estación')
ejes[1].plot(periodo_grafico['fecha'], periodo_grafico['VPD_FAO_ERA5_kPa'],
             color='#7570b3', label='VPD FAO ERA5-Land')
ejes[1].set_ylabel('VPD (kPa)'); ejes[1].set_xlabel('Fecha'); ejes[1].legend()
plt.tight_layout(); plt.show()

### Interpretación de la validación

- **R² o r:** indica sincronía temporal, no igualdad de valores.
- **RMSE y MAE:** expresan el tamaño del error en las unidades de la variable.
- **Sesgo y PBIAS:** indican sobreestimación o subestimación sistemática.
- **KGE:** combina correlación, variabilidad y sesgo medio; debe interpretarse junto con las demás métricas.
- Una corrección estadística obtenida en Colliguay no se transfiere automáticamente a Casablanca. En esta actividad se conserva ERA5-Land **sin calibración** y la validación se usa para conocer su desempeño.

## 9. Aplicar ETo y VPD de ERA5-Land en Casablanca

Se calcula la media de ERA5-Land sobre el área monitoreada. Dado que el AOI es mucho menor que una celda ERA5-Land, la serie representa la condición meteorológica regional. No describe diferencias entre hileras o parcelas.

In [ ]:
meteo_casablanca = extraer_era5(
    aoi, ANALISIS_INICIO, ANALISIS_FIN_EXCLUSIVA, ee.Reducer.mean()
)
print('Días ERA5-Land en Casablanca:', len(meteo_casablanca))
display(meteo_casablanca.head())

fig, ejes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ejes[0].plot(meteo_casablanca['fecha'], meteo_casablanca['ETo_ERA5_mm_dia'], color='#d95f02')
ejes[0].set_ylabel('ETo (mm/día)'); ejes[0].set_title('Demanda atmosférica regional en Casablanca')
for columna, color in [
    ('VPD_medio_ERA5_kPa', '#1b9e77'), ('VPD_FAO_ERA5_kPa', '#7570b3'),
    ('VPD_max_ERA5_kPa', '#e7298a')
]:
    ejes[1].plot(meteo_casablanca['fecha'], meteo_casablanca[columna], label=columna, color=color)
ejes[1].set_ylabel('VPD (kPa)'); ejes[1].set_xlabel('Fecha'); ejes[1].legend(ncol=3, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
imagen_meteo = (
    ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
    .filterDate(FECHA_MAPA, ee.Date(FECHA_MAPA).advance(1, 'day'))
    .select(ERA5_BANDAS).map(agregar_eto_vpd_era5).first()
)
imagen_meteo = ee.Image(imagen_meteo)

mapa_meteo = geemap.Map(basemap={})
mapa_meteo.centerObject(aoi, 9)
mapa_meteo.addLayer(
    imagen_meteo.select('ETo_ERA5_mm_dia').clip(aoi.buffer(30000)),
    {'min': 0, 'max': 8, 'palette': ['ffffcc', 'fd8d3c', '800026']},
    f'ETo ERA5-Land {FECHA_MAPA}', True
)
mapa_meteo.addLayer(
    imagen_meteo.select('VPD_max_ERA5_kPa').clip(aoi.buffer(30000)),
    {'min': 0, 'max': 4, 'palette': ['2c7bb6', 'ffffbf', 'd7191c']},
    f'VPD máximo ERA5-Land {FECHA_MAPA}', False
)
mapa_meteo.addLayer(aoi_fc.style(color='000000', fillColor='00000000', width=3), {}, 'Área de Casablanca')
mapa_meteo

## 10. Sentinel-2: misma filtración de la Clase 1

Se utiliza `COPERNICUS/S2_SR_HARMONIZED`, filtro espacial, período de análisis, límite de nubosidad de la escena y máscara con la banda SCL. Se excluyen sombras de nube, nubes medias/altas, cirros, píxeles saturados y sin datos.

In [ ]:
s2_bruto = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(ANALISIS_INICIO, ANALISIS_FIN_EXCLUSIVA)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', NUBOSIDAD_MAX))
)

def preparar_s2(imagen):
    imagen = ee.Image(imagen)
    scl = imagen.select('SCL')
    mascara = (
        scl.neq(0).And(scl.neq(1)).And(scl.neq(3))
        .And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    )
    reflectancia = imagen.select(['B3', 'B4', 'B8', 'B11']).multiply(0.0001)
    return reflectancia.updateMask(mascara).copyProperties(
        imagen, ['system:time_start', 'CLOUDY_PIXEL_PERCENTAGE']
    )

s2 = s2_bruto.map(preparar_s2)
print('Escenas antes de la máscara:', s2_bruto.size().getInfo())
print('Escenas preparadas:', s2.size().getInfo())
if s2.size().getInfo() == 0:
    raise RuntimeError('No hay escenas Sentinel-2. Amplíe fechas o nubosidad máxima.')

## 11. Índices espectrales

| Índice | Ecuación | Lectura general |
|---|---|---|
| NDVI | (B8 − B4)/(B8 + B4) | vigor y cantidad de vegetación verde |
| NDMI | (B8 − B11)/(B8 + B11) | contenido de humedad del dosel; valores mayores suelen indicar más humedad |
| MSI | B11/B8 | estrés o menor contenido de agua; valores mayores suelen indicar mayor estrés |
| DWSI | (B8 + B3)/(B11 + B4) | condición hídrica/sanitaria combinada; valores mayores suelen asociarse con mejor condición |

MSI y NDMI utilizan las mismas bandas y estarán matemáticamente relacionados. Ninguno mide directamente el potencial hídrico del tallo. Su interpretación depende de especie, suelo, fenología, arquitectura del dosel y calibración local.

In [ ]:
INDICES = ['NDVI', 'NDMI', 'MSI', 'DWSI']

def agregar_indices(imagen):
    imagen = ee.Image(imagen)
    b3, b4 = imagen.select('B3'), imagen.select('B4')
    b8, b11 = imagen.select('B8'), imagen.select('B11')
    ndvi = b8.subtract(b4).divide(b8.add(b4)).rename('NDVI')
    ndmi = b8.subtract(b11).divide(b8.add(b11)).rename('NDMI')
    msi = b11.divide(b8).rename('MSI')
    dwsi = b8.add(b3).divide(b11.add(b4)).rename('DWSI')
    return ee.Image.cat([ndvi, ndmi, msi, dwsi]).copyProperties(
        imagen, ['system:time_start', 'CLOUDY_PIXEL_PERCENTAGE']
    )

indices_s2 = ee.ImageCollection(s2.map(agregar_indices))
print('Bandas:', ee.Image(indices_s2.first()).bandNames().getInfo())

In [ ]:
compuesto = indices_s2.median().clip(aoi)
VIS = {
    'NDVI': {'min': 0, 'max': 0.9, 'palette': ['8c510a', 'f6e8c3', '1a9850']},
    'NDMI': {'min': -0.4, 'max': 0.7, 'palette': ['a6611a', 'f5f5f5', '018571']},
    'MSI': {'min': 0.2, 'max': 1.5, 'palette': ['2166ac', 'f7f7f7', 'b2182b']},
    'DWSI': {'min': 0.5, 'max': 4, 'palette': ['b2182b', 'f7f7f7', '1a9850']},
}

mapa_indices = geemap.Map(basemap={})
mapa_indices.centerObject(aoi, 16)
for indice in INDICES:
    mapa_indices.addLayer(compuesto.select(indice), VIS[indice], f'{indice} mediana', indice == 'MSI')
mapa_indices.addLayer(aoi_fc.style(color='FFCC00', fillColor='00000000', width=2), {}, 'Área')
mapa_indices

In [ ]:
histograma = compuesto.reduceRegion(
    reducer=ee.Reducer.toList(), geometry=aoi, scale=20,
    bestEffort=True, maxPixels=1e6,
).getInfo()

fig, ejes = plt.subplots(1, 4, figsize=(16, 3.8))
for eje, indice, color in zip(ejes, INDICES, ['#4d9221', '#2166ac', '#b2182b', '#762a83']):
    valores = np.asarray(histograma.get(indice, []), dtype=float)
    valores = valores[np.isfinite(valores)]
    if valores.size:
        eje.hist(valores, bins=20, color=color, alpha=0.8)
        eje.axvline(np.median(valores), color='black', ls='--', label='Mediana')
        eje.legend(fontsize=8)
    else:
        eje.text(0.5, 0.5, 'Sin píxeles válidos', ha='center', va='center', transform=eje.transAxes)
    eje.set_title(indice); eje.set_xlabel('Valor'); eje.set_ylabel('Píxeles')
plt.tight_layout(); plt.show()

print('Observe amplitud, asimetría, valores extremos y diferencias entre índices.')

## 12. Serie temporal de índices

Cada imagen se resume mediante la mediana espacial del área. Las escenas sin píxeles válidos se eliminan después de la extracción.

In [ ]:
def resumir_indices(imagen):
    imagen = ee.Image(imagen)
    valores = imagen.select(INDICES).reduceRegion(
        reducer=ee.Reducer.median(), geometry=aoi, scale=20,
        bestEffort=True, maxPixels=1e7,
    )
    return ee.Feature(None, valores).set({
        'fecha': imagen.date().format('YYYY-MM-dd'),
        'nubosidad_escena': imagen.get('CLOUDY_PIXEL_PERCENTAGE'),
    })

serie_indices = geemap.ee_to_df(ee.FeatureCollection(indices_s2.map(resumir_indices)))
serie_indices['fecha'] = pd.to_datetime(serie_indices['fecha'])
for columna in INDICES + ['nubosidad_escena']:
    serie_indices[columna] = pd.to_numeric(serie_indices[columna], errors='coerce')
serie_indices = serie_indices.dropna(subset=INDICES).sort_values('fecha')

fig, ejes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for eje, indice, color in zip(ejes.ravel(), INDICES, ['#4d9221', '#2166ac', '#b2182b', '#762a83']):
    eje.plot(serie_indices['fecha'], serie_indices[indice], 'o-', color=color)
    eje.set_title(indice); eje.set_ylabel('Mediana del área')
    eje.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

print('Adquisiciones válidas:', len(serie_indices))
display(serie_indices.head())

## 13. Condiciones antecedentes de 3 y 7 días

La vegetación puede responder con retraso. Para cada fecha Sentinel-2 se comparan:

- condición meteorológica del mismo día;
- ETo acumulada durante los 3 y 7 días terminados en la fecha de la imagen;
- VPD medio durante los 3 y 7 días terminados en la fecha de la imagen;
- VPD máximo observado dentro de esas ventanas.

Las ventanas incluyen el día Sentinel-2 y los días inmediatamente anteriores. No se rellenan escenas ópticas ausentes.

In [ ]:
meteo_ventanas = meteo_casablanca.sort_values('fecha').set_index('fecha').copy()
for dias in [3, 7]:
    meteo_ventanas[f'ETo_{dias}d_suma_mm'] = (
        meteo_ventanas['ETo_ERA5_mm_dia'].rolling(dias, min_periods=dias).sum()
    )
    meteo_ventanas[f'VPD_FAO_{dias}d_media_kPa'] = (
        meteo_ventanas['VPD_FAO_ERA5_kPa'].rolling(dias, min_periods=dias).mean()
    )
    meteo_ventanas[f'VPD_max_{dias}d_max_kPa'] = (
        meteo_ventanas['VPD_max_ERA5_kPa'].rolling(dias, min_periods=dias).max()
    )
meteo_ventanas = meteo_ventanas.reset_index()

integrada = (
    serie_indices.merge(meteo_ventanas, on='fecha', how='inner')
    .dropna(subset=INDICES + ['ETo_ERA5_mm_dia', 'VPD_FAO_ERA5_kPa'])
    .sort_values('fecha')
)

columnas_mostrar = [
    'fecha', 'NDVI', 'NDMI', 'MSI', 'DWSI', 'ETo_ERA5_mm_dia',
    'VPD_medio_ERA5_kPa', 'VPD_FAO_ERA5_kPa', 'VPD_max_ERA5_kPa',
    'ETo_3d_suma_mm', 'ETo_7d_suma_mm', 'VPD_FAO_7d_media_kPa',
]
print('Fechas Sentinel-2 integradas:', len(integrada))
display(integrada[columnas_mostrar].head())

## 14. Correlaciones: contemporáneas y antecedentes

Se muestran dos coeficientes:

- **Pearson:** cuantifica asociación lineal.
- **Spearman:** cuantifica asociación monotónica y es menos sensible a valores extremos.

La muestra está limitada por las fechas Sentinel-2 sin nubes. Una correlación puede reflejar estacionalidad compartida y no demuestra causalidad.

In [ ]:
predictores_meteo = [
    'ETo_ERA5_mm_dia', 'VPD_medio_ERA5_kPa', 'VPD_FAO_ERA5_kPa',
    'VPD_max_ERA5_kPa', 'ETo_3d_suma_mm', 'ETo_7d_suma_mm',
    'VPD_FAO_3d_media_kPa', 'VPD_FAO_7d_media_kPa',
    'VPD_max_3d_max_kPa', 'VPD_max_7d_max_kPa',
]
columnas_corr = INDICES + predictores_meteo
corr_pearson = integrada[columnas_corr].corr(method='pearson').loc[INDICES, predictores_meteo]
corr_spearman = integrada[columnas_corr].corr(method='spearman').loc[INDICES, predictores_meteo]

fig, ejes = plt.subplots(2, 1, figsize=(13, 7.5))
sns.heatmap(corr_pearson, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', ax=ejes[0])
ejes[0].set_title('Correlación de Pearson')
sns.heatmap(corr_spearman, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', ax=ejes[1])
ejes[1].set_title('Correlación de Spearman')
plt.tight_layout(); plt.show()

print('Compare si las asociaciones cambian entre el mismo día y las ventanas de 3 o 7 días.')

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(12, 9))
pares = [
    ('MSI', 'VPD_FAO_ERA5_kPa'), ('MSI', 'VPD_FAO_7d_media_kPa'),
    ('DWSI', 'ETo_ERA5_mm_dia'), ('DWSI', 'ETo_7d_suma_mm'),
]
for eje, (x, y) in zip(ejes.ravel(), pares):
    sns.regplot(data=integrada, x=x, y=y, ax=eje,
                scatter_kws={'s': 50, 'alpha': 0.7}, line_kws={'color': '#333333'})
    r = integrada[[x, y]].corr(method='spearman').iloc[0, 1]
    eje.set_title(f'{x} vs. {y} | Spearman={r:.2f}')
plt.tight_layout(); plt.show()

## 15. Comparar trayectorias estandarizadas

Los índices, la ETo y el VPD tienen unidades y amplitudes diferentes. El puntaje estandarizado \(z\) resta la media y divide por la desviación estándar. Permite comparar sincronía y dirección, sin convertir las variables a una misma magnitud física.

In [ ]:
comparar = ['MSI', 'DWSI', 'ETo_ERA5_mm_dia', 'VPD_FAO_ERA5_kPa']
z = integrada[['fecha']].copy()
for coluna in comparar:
    desviacion = integrada[coluna].std(ddof=0)
    z[coluna] = (integrada[coluna] - integrada[coluna].mean()) / desviacion if desviacion > 0 else np.nan

fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for coluna, color in [('MSI', '#b2182b'), ('ETo_ERA5_mm_dia', '#d95f02'), ('VPD_FAO_ERA5_kPa', '#7570b3')]:
    ejes[0].plot(z['fecha'], z[coluna], 'o-', label=coluna, color=color)
for columna, color in [('DWSI', '#4d9221'), ('ETo_ERA5_mm_dia', '#d95f02'), ('VPD_FAO_ERA5_kPa', '#7570b3')]:
    ejes[1].plot(z['fecha'], z[columna], 'o-', label=columna, color=color)
for eje, titulo in zip(ejes, ['MSI y demanda atmosférica', 'DWSI y demanda atmosférica']):
    eje.axhline(0, color='gray', lw=0.8); eje.set_title(titulo); eje.legend(fontsize=8)
    eje.tick_params(axis='x', rotation=40)
ejes[0].set_ylabel('Valor estandarizado (z)')
plt.tight_layout(); plt.show()

## 16. Guardar resultados

Se guardan cuatro tablas:

1. pares diarios de validación en Colliguay;
2. métricas de validación;
3. meteorología ERA5-Land diaria de Casablanca;
4. tabla integrada por fecha Sentinel-2 para continuar en la TP4.

In [ ]:
ruta_validacion = CARPETA_SALIDA / 'validacion_ERA5_Colliguay_diaria.csv'
ruta_metricas = CARPETA_SALIDA / 'metricas_validacion_ERA5_Colliguay.csv'
ruta_meteo = CARPETA_SALIDA / 'meteorologia_ERA5_Casablanca_diaria.csv'
ruta_integrada = CARPETA_SALIDA / 'tabla_TP3_indices_ETo_VPD_Casablanca.csv'

validacion.to_csv(ruta_validacion, index=False)
tabla_metricas.reset_index().to_csv(ruta_metricas, index=False)
meteo_ventanas.to_csv(ruta_meteo, index=False)
integrada.to_csv(ruta_integrada, index=False)

for ruta in [ruta_validacion, ruta_metricas, ruta_meteo, ruta_integrada]:
    print('Guardado:', ruta)

## 17. Preguntas guía para interpretar los resultados

1. ¿Qué diferencias existen entre una medición puntual y una celda ERA5-Land de aproximadamente 11 km?
2. ¿ERA5-Land reproduce mejor la variación temporal o la magnitud absoluta de la ETo en Colliguay?
3. ¿Qué información aportan RMSE, PBIAS y KGE que no aparece solamente en \(R^2\)?
4. ¿Por qué el VPD máximo suele superar al VPD medio y al VPD FAO?
5. ¿Cuál de los métodos ERA5-Land se aproxima mejor a su equivalente calculado con la estación?
6. ¿Es correcto afirmar que el VPD fue medido directamente en Colliguay? Explique.
7. ¿MSI y NDMI cambian en sentidos opuestos? ¿Qué relación matemática explica este comportamiento?
8. ¿Las relaciones con ETo o VPD son más claras el mismo día o usando los 3–7 días anteriores?
9. ¿Pearson y Spearman producen conclusiones semejantes? ¿Qué puede explicar sus diferencias?
10. ¿Qué observación de campo sería necesaria para relacionar estos índices con estrés hídrico real del cultivo?

## 18. Interpretación responsable y límites

- ETo representa demanda evaporativa de una superficie de referencia; no es ETa del cultivo.
- VPD representa demanda atmosférica, pero no informa por sí solo cuánta agua está disponible en el suelo.
- El VPD de Colliguay se calcula a partir de temperatura y humedad relativa; no es una observación directa de VPD.
- ERA5-Land describe condiciones regionales y no debe interpretarse como variación dentro del huerto.
- MSI, NDMI, DWSI y NDVI responden también a fenología, suelo visible, estructura del dosel y prácticas de manejo.
- MSI y NDMI comparten NIR y SWIR, por lo que parte de su correlación es matemática.
- La nubosidad reduce y sesga el muestreo temporal de Sentinel-2.
- Una correlación temporal puede estar impulsada por la estación del año.
- No deben aplicarse umbrales universales de estrés sin calibración con datos de campo.

## Referencias y fuentes

- Zambrano et al. (2025). *Prediction of the daily spatial variation of stem water potential in cherry orchards using weather and Sentinel-2 data*.
- Allen, R. G. et al. (1998). *Crop evapotranspiration: Guidelines for computing crop water requirements*. FAO Irrigation and Drainage Paper 56.
- ERA5-Land Daily Aggregated: https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_DAILY_AGGR
- Sentinel-2 SR Harmonized: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED
- Agromet Chile: https://agrometeorologia.cl/

**Próxima actividad:** en la TP4, la tabla integrada podrá combinarse con Sentinel-1 y otros predictores para un ejercicio de Machine Learning orientado a ETa.